In [14]:
import csv
import pandas as pd
from datetime import datetime
import numpy as np
import math


#Data Collection
corporate_bond_spread = pd.read_csv('data/corporate_bond_spread_csv.csv') #Header: {Date, Value} Shape: (7634, 2) 
corporate_bond_spread = corporate_bond_spread.rename(columns={'Date': 'observation_date'})

FRED_yield_spread = pd.read_csv('data/FRED_high_yield_index_spread_csv.csv') #Header: {observation_date, BAMLH0A0HYM2} Shape: (2644, 2) Start Date: 2016-03-30
unemployment = pd.read_csv('data/unemployment_csv.csv') #Header: {observation_date, UNRATE} #Shape: (120, 2) Start date: 2016-03-01

market_rates = pd.read_csv('data/fed_funds_rate_csv.csv') 
market_rates = market_rates.rename(columns={'Effective Date' : 'observation_date'})
market_rates = market_rates[(market_rates['observation_date'].notna())]
'''Index(['Effective Date', 'Rate Type', 'Rate (%)', '1st Percentile (%)',
       '25th Percentile (%)', '75th Percentile (%)', '99th Percentile (%)',
       'Volume ($Billions)', 'Target Rate From (%)', 'Target Rate To (%)',
       'Intra Day - Low (%)', 'Intra Day - High (%)', 'Standard Deviation (%)',
       '30-Day Average SOFR', '90-Day Average SOFR', '180-Day Average SOFR',
       'SOFR Index', 'Revision Indicator (Y/N)', 'Footnote ID'],
      dtype='str')

      Shape: (11051, 19)

      '''
t_bill_10y = pd.read_csv('data/DGS10.csv')
t_bill_2y = pd.read_csv('data/DGS2.csv')
corp_bond_spread = pd.read_csv('data/corporate_bond_spread_csv.csv')
b_a = pd.read_csv('data/BAA_AAA.csv')
#Data Normalization
data = [b_a, t_bill_10y, t_bill_2y]

In [7]:
for dataset in data:
    min_date = dataset['observation_date'].iloc[0]
    max_date = dataset['observation_date'].iloc[-1]
    print(min_date, max_date)

1919-01-01 2026-03-01
1962-01-02 2026-04-06
1976-06-01 2026-04-06


In [15]:
normal_format_string = "%Y-%m-%d"
start = datetime.strptime('1962-01-02', normal_format_string)
end = datetime.strptime('2026-03-01', normal_format_string)


In [5]:
print(corporate_bond_spread)

     observation_date  Value
0          12/31/1996   0.60
1          01/02/1997   0.60
2          01/03/1997   0.61
3          01/06/1997   0.61
4          01/07/1997   0.61
...               ...    ...
7629       03/23/2026   0.88
7630       03/24/2026   0.87
7631       03/25/2026   0.87
7632       03/26/2026   0.88
7633       03/27/2026   0.91

[7634 rows x 2 columns]


In [16]:

# normal_format_string = "%Y-%m-%d"
# strange_format_string = "%m/%d/%Y"

# start = datetime.strptime('2016-03-01', normal_format_string)
# end = datetime.strptime('2026-02-01', normal_format_string)

# strange_format_string = [corporate_bond_spread, market_rates]
# normal_format_string = [FRED_yield_spread, unemployment]

for dataset in data:
   
    dataset['observation_date'] = pd.to_datetime(dataset['observation_date'])
    dataset = dataset[(dataset['observation_date'] >= start) ]
    dataset = dataset[(dataset['observation_date'] <= end) ]
    print(dataset.shape)
#Features


(770, 2)
(16739, 2)
(12979, 2)


In [17]:
for dataset in data:
    print(dataset)

     observation_date  BAA_AAA
0          1919-01-01     1.77
1          1919-02-01     1.85
2          1919-03-01     1.76
3          1919-04-01     1.79
4          1919-05-01     1.70
...               ...      ...
1282       2025-11-01     0.60
1283       2025-12-01     0.59
1284       2026-01-01     0.54
1285       2026-02-01     0.51
1286       2026-03-01     0.56

[1287 rows x 2 columns]
      observation_date  DGS10
0           1962-01-02   4.06
1           1962-01-03   4.03
2           1962-01-04   3.99
3           1962-01-05   4.02
4           1962-01-08   4.03
...                ...    ...
16760       2026-03-31   4.30
16761       2026-04-01   4.33
16762       2026-04-02   4.31
16763       2026-04-03   4.35
16764       2026-04-06   4.34

[16765 rows x 2 columns]
      observation_date  DGS2
0           1976-06-01  7.26
1           1976-06-02  7.23
2           1976-06-03  7.22
3           1976-06-04  7.12
4           1976-06-07  7.09
...                ...   ...
13000       20

In [18]:

#Corporate bond minus FRED yield

df = pd.DataFrame()
# print(corporate_bond_spread, FRED_yield_spread)
# test = pd.merge(corporate_bond_spread, FRED_yield_spread, on='observation_date', suffixes=('_df1', '_df2'))
test2 = pd.merge(t_bill_10y, t_bill_2y, on='observation_date', suffixes=('_df1', '_df2'))
# # print(test2)
# df['observation_date'] = test['observation_date']
# df['Premium_Junk_Spread'] = test['BAMLH0A0HYM2'] - test['Value']
df['observation_date'] = t_bill_10y['observation_date']
df['Premium_BBB_Spread'] = b_a['BAA_AAA']
df['10Y-2Y_spread'] = test2['DGS10'] - test2['DGS2']
print(df)

      observation_date  Premium_BBB_Spread  10Y-2Y_spread
0           1962-01-02                1.77           0.68
1           1962-01-03                1.85           0.71
2           1962-01-04                1.76           0.70
3           1962-01-05                1.79           0.77
4           1962-01-08                1.70           0.79
...                ...                 ...            ...
16760       2026-03-31                 NaN            NaN
16761       2026-04-01                 NaN            NaN
16762       2026-04-02                 NaN            NaN
16763       2026-04-03                 NaN            NaN
16764       2026-04-06                 NaN            NaN

[16765 rows x 3 columns]


In [14]:
#Sahm rule: Signals a recession when 3-month moving average of US unemployment rate rises by 0.5 percentage points or more above lowest three-month 
#average over the past year --> lagging indicator b/c unemployment is lagging 



In [19]:
#Lets try predicting recession with only these two features

clean_df = df[df['Premium_BBB_Spread'].notna()]
print(clean_df)

     observation_date  Premium_BBB_Spread  10Y-2Y_spread
0          1962-01-02                1.77           0.68
1          1962-01-03                1.85           0.71
2          1962-01-04                1.76           0.70
3          1962-01-05                1.79           0.77
4          1962-01-08                1.70           0.79
...               ...                 ...            ...
1282       1966-12-01                0.60          -0.73
1283       1966-12-02                0.59          -0.67
1284       1966-12-05                0.54          -0.90
1285       1966-12-06                0.51          -1.25
1286       1966-12-07                0.56          -1.27

[1287 rows x 3 columns]
